# NaviKurd: 80 everyday object categories
Run top to bottom in a GPU runtime with persistent storage with internet access. Source: [COCO 2017](https://cocodataset.org/#download). Download official annotations and only the selected training photographs. Budget tens of GB of disk space for 100,000 distinct photos and persist the dataset and training runs between sessions. This is a course experiment, not a navigation aid.

In [ ]:
%pip -q install ultralytics==8.3.228

In [ ]:
import json, random, zipfile, shutil, yaml, csv
from pathlib import Path
from urllib.request import urlretrieve
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict, Counter
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO
root = Path('navikurd_data/coco100k_all80_v1').resolve(); root.mkdir(parents=True, exist_ok=True)
z = root/'annotations.zip'
if not z.exists(): urlretrieve('https://s3.amazonaws.com/images.cocodataset.org/annotations/annotations_trainval2017.zip', z)
with zipfile.ZipFile(z) as f: data = json.load(f.open('annotations/instances_train2017.json'))
categories = sorted(data['categories'], key=lambda x: x['id'])
names = [x['name'] for x in categories]
classes = {x['id']: k for k, x in enumerate(categories)}
assert len(classes) == len(names) == 80
print(', '.join(names))
boxes = defaultdict(list); crowd = set()
for a in data['annotations']:
    if a['category_id'] in classes:
        if a['iscrowd']: crowd.add(a['image_id'])
        else: boxes[a['image_id']].append((classes[a['category_id']], a['bbox']))
images = {x['id']:x for x in data['images']}; del data

Sample 100,000 unique photos without replacement from images without crowd annotations. Keep images with no annotated objects eligible as background. Split by image ID, not by box.

In [ ]:
rng = random.Random(42)
eligible = sorted(images.keys() - crowd)
total = 100_000
assert len(eligible) >= total
ids = rng.sample(eligible, total)
splits = dict(train=ids[:70000], val=ids[70000:85000], test=ids[85000:])
assert len(set(ids)) == len(ids) == total
for split, subset in splits.items():
    counts = Counter(names[k] for i in subset for k,_ in boxes.get(i, []))
    print(split, len(subset), 'images;', sum(not boxes.get(i) for i in subset), 'background images;', counts)
    assert len(counts) == len(names)
(root/'split_ids.json').write_text(json.dumps(splits))

In [ ]:
def prepare(job):
    split,i=job; info=images[i]; w,h=info['width'],info['height']; filename=info['file_name']
    img=root/'images'/split/filename; label=root/'labels'/split/(img.stem+'.txt')
    if not img.exists():
        tmp=img.with_suffix('.part'); urlretrieve('https://s3.amazonaws.com/images.cocodataset.org/train2017/'+filename,tmp); tmp.replace(img)
    rows=[]
    for k,(x,y,bw,bh) in boxes[i]:
        rows.append(f'{k} {(x+bw/2)/w} {(y+bh/2)/h} {bw/w} {bh/h}')
    label.write_text('\n'.join(rows))
for s in splits:
    for d in ['images','labels']: (root/d/s).mkdir(parents=True,exist_ok=True)
with ThreadPoolExecutor(6) as pool: list(pool.map(prepare,[(s,i) for s in splits for i in splits[s]]))
cfg=root/'coco80.yaml'; cfg.write_text(yaml.safe_dump(dict(path=str(root),names=names,**{s:'images/'+s for s in splits})))

In [ ]:
from matplotlib.patches import Rectangle
fig,axs=plt.subplots(2,5,figsize=(16,7))
for ax,i in zip(axs.flat,[i for i in splits['train'] if boxes.get(i)][:10]):
    ax.imshow(Image.open(root/'images/train'/images[i]['file_name'])); ax.axis('off')
    for k,(x,y,w,h) in boxes[i]:
        ax.add_patch(Rectangle((x,y),w,h,fill=False,color='red')); ax.text(x,y,names[k],backgroundcolor='white')
plt.tight_layout()

Train a small YOLO detector from random initialization to avoid COCO pretraining overlap. Validation selects the best epoch; inspect loss curves for overfitting. Inspect per-class results because common classes can hide poor results on rare ones.

In [ ]:
model=YOLO('yolo11n.yaml'); model.train(data=str(cfg),epochs=40,patience=8,imgsz=416,batch=8,seed=42,workers=2,pretrained=False)
run=Path(model.trainer.save_dir); display(Image.open(run/'results.png'))
best=YOLO(str(run/'weights/best.pt'))

Final held-out evaluation: run after freezing all settings. AP measures detection quality across confidence thresholds. Discuss weak classes and example errors in your report; do not tune on these results.

In [ ]:
scores=best.val(data=str(cfg),split='test',imgsz=416,plots=True); print('Test mAP50-95:',scores.box.map)
for k,ap in zip(scores.box.ap_class_index,scores.box.ap): print(names[k],float(ap))
i=splits['test'][0]; prediction=best(str(root/'images/test'/images[i]['file_name']))[0]; plt.imshow(prediction.plot()[:,:,::-1]); plt.axis('off');

## Camera demo (Google Colab, one snapshot on demand)
After training, run the next cell and allow camera access. Point the camera, then activate **Describe this view**. The frame is sent to your Colab runtime for detection; this cell does not save it. The camera stops after capture or cancellation. The description is displayed and spoken if your browser supports speech. For a fresh session, run the package installation cell, upload your own trained `best.pt` to `/content/best.pt`, then run this demo without downloading COCO. Only photograph people with permission. This is not a navigation aid.

In [ ]:
import base64, io, json
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
from google.colab.output import eval_js

if 'best' not in globals():
    checkpoint = Path('/content/best.pt')
    if not checkpoint.is_file():
        raise FileNotFoundError('Upload your trained best.pt to /content or run the training cell first.')
    best = YOLO(str(checkpoint))

snapshot = eval_js("""(async () => {
  const panel = document.createElement('div');
  const video = document.createElement('video');
  video.muted = true; video.playsInline = true; video.width = 320;
  const snap = document.createElement('button'); snap.textContent = 'Describe this view';
  const cancel = document.createElement('button'); cancel.textContent = 'Cancel';
  let stream;
  try {
    stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: {ideal: 'environment'}}, audio: false});
    video.srcObject = stream; panel.append(video, snap, cancel); document.body.append(panel);
    await video.play(); google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
    snap.focus();
    if (!await new Promise(resolve => {snap.onclick = () => resolve(true); cancel.onclick = () => resolve(false);})) return null;
    const canvas = document.createElement('canvas');
    canvas.width = video.videoWidth; canvas.height = video.videoHeight;
    canvas.getContext('2d').drawImage(video, 0, 0);
    return canvas.toDataURL('image/jpeg', 0.85);
  } catch (err) {throw new Error('Camera unavailable: check browser camera permission. ' + err.message);}
  finally {if (stream) stream.getTracks().forEach(track => track.stop()); panel.remove();}
})()""")
if snapshot:
    frame = Image.open(io.BytesIO(base64.b64decode(snapshot.split(',', 1)[1]))).convert('RGB')
    result = best.predict(frame, imgsz=416, verbose=False)[0]
    phrases = []
    for box in sorted(result.boxes, key=lambda b: float(b.conf.item()), reverse=True):
        x = float(box.xywhn[0, 0]); side = 'on the left' if x < 1/3 else 'on the right' if x > 2/3 else 'in the center'
        phrase = f'{result.names[int(box.cls.item())]} {side}'
        if phrase not in phrases: phrases.append(phrase)
        if len(phrases) == 3: break
    message = '; '.join(phrases) + '.' if phrases else "I couldn't identify an object."
    print(message); display(Image.fromarray(result.plot()[:, :, ::-1]))
    eval_js("if ('speechSynthesis' in window) { window.speechSynthesis.cancel(); window.speechSynthesis.speak(new SpeechSynthesisUtterance(" + json.dumps(message) + ")); }")
else:
    print('Capture cancelled.')